# 💳 Credit Card Fraud Detection Pipeline
### End-to-End Machine Learning with Python, Pandas, Scikit-learn, and SMOTE

---

## 📌 Project Objectives
1. **Analyze and Preprocess 284,000+ Transactions**: Conduct in-depth EDA, extract temporal and financial features, and apply `RobustScaler` to mitigate outlier distortion.
2. **Mitigate Severe Class Imbalance with SMOTE**: Address the extreme 577:1 imbalance (0.172% fraud rate) exclusively on the training split to eliminate data snooping.
3. **Train Classification Models**: Train **Logistic Regression** and **Random Forest Classifier** with stratified cross-validation.
4. **Evaluate Performance**: Measure **Precision, Recall, F1-Score, ROC-AUC, PR-AUC, and Confusion Matrices**.
5. **Identify Key Fraud Predictors**: Analyze Random Forest feature importances and Logistic Regression coefficients to explain model decisions.


In [ ]:
# Step 1: Environment Setup and Library Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid')
print('Libraries imported successfully!')

## 📥 2. Loading the Credit Card Transactions Dataset
The dataset comprises **284,807 transactions** collected over 48 hours. Features V1–V28 represent PCA components; `Time`, `Amount`, and `Class` (0 = Legitimate, 1 = Fraud) represent the transaction metadata.

In [ ]:
data_path = Path('../data/creditcard.csv')
if not data_path.exists():
    # Fallback to local import from src
    sys.path.append('..')
    from src.data_loader import load_creditcard_data
    df = load_creditcard_data(data_path)
else:
    df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())

## 📊 3. Exploratory Data Analysis (EDA)
Let's inspect the class imbalance, transaction amounts, and transaction timing.

In [ ]:
# Class Imbalance Check
class_counts = df['Class'].value_counts()
print('Class Counts:')
print(class_counts)
print(f"\nFraud Percentage: {(class_counts[1] / len(df)) * 100:.3f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_yscale('log')
axes[0].set_title('Transaction Counts (Logarithmic Scale)', fontweight='bold')
axes[0].set_ylabel('Log Count')

axes[1].pie(class_counts.values, labels=['Legitimate (0)', 'Fraud (1)'], autopct='%1.3f%%', colors=['#2ecc71', '#e74c3c'], explode=(0, 0.2))
axes[1].set_title('Class Imbalance Proportion', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Amount and Time Distribution
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, ax=ax1, color='#2ecc71', kde=True)
ax1.set_title('Legitimate Transaction Amount Distribution (Log Scale)', fontweight='bold')
ax1.set_yscale('log')

sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, ax=ax2, color='#e74c3c', kde=True)
ax2.set_title('Fraudulent Transaction Amount Distribution (Log Scale)', fontweight='bold')
ax2.set_yscale('log')
ax2.set_xlabel('Amount ($)')
plt.tight_layout()
plt.show()

## ⚙️ 4. Feature Engineering & Feature Scaling
- Extract cyclical `Hour_of_Day` from `Time` seconds.
- Compute `Log_Amount` = $\log(1 + \text{Amount})$.
- Split into **Stratified Train-Test** partitions (80% Train, 20% Test).
- Apply `RobustScaler` to `Amount` and `Time` using median and IQR, fitting **strictly on the training partition**.

In [ ]:
# Feature Engineering
df_feat = df.copy()
df_feat['Hour_of_Day'] = ((df_feat['Time'] // 3600) % 24).astype(int)
df_feat['Log_Amount'] = np.log1p(df_feat['Amount'])

X = df_feat.drop(columns=['Class'])
y = df_feat['Class']

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Robust Feature Scaling
scaler = RobustScaler()
scaled_train = scaler.fit_transform(X_train[['Amount', 'Time']])
scaled_test = scaler.transform(X_test[['Amount', 'Time']])

X_train = X_train.copy()
X_test = X_test.copy()
X_train['scaled_amount'] = scaled_train[:, 0]
X_train['scaled_time'] = scaled_train[:, 1]
X_test['scaled_amount'] = scaled_test[:, 0]
X_test['scaled_time'] = scaled_test[:, 1]

X_train = X_train.drop(columns=['Amount', 'Time'])
X_test = X_test.drop(columns=['Amount', 'Time'])

print(f'Training shape: {X_train.shape} | Fraud count: {y_train.sum()}')
print(f'Testing shape:  {X_test.shape}  | Fraud count: {y_test.sum()}')

## ⚖️ 5. Handling Severe Class Imbalance with SMOTE
We apply Synthetic Minority Over-sampling Technique (**SMOTE**) strictly on the training set to generate synthetic fraud instances and establish balanced decision boundaries.

In [ ]:
sys.path.append('..')
from src.smote_handler import apply_smote

X_train_resampled, y_train_resampled = apply_smote(X_train, y_train)
print(f'Resampled training instances: {len(X_train_resampled):,}')

## 🤖 6. Model Training: Logistic Regression & Random Forest
We train two diverse paradigms:
1. **Logistic Regression**: Interpretable linear classifier with L2 regularization.
2. **Random Forest Classifier**: Non-linear ensemble of decision trees capturing complex feature interactions.

In [ ]:
# Train Logistic Regression
print('Training Logistic Regression...')
lr_model = LogisticRegression(C=0.1, max_iter=1000, solver='lbfgs', random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)
print('Logistic Regression training complete!')

# Train Random Forest Classifier
print('Training Random Forest Classifier...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=14, min_samples_split=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train_resampled, y_train_resampled)
print('Random Forest training complete!')

## 📈 7. Model Evaluation on Untouched Test Split
We evaluate both models using **Precision, Recall, F1-Score, ROC-AUC, PR-AUC**, and **Confusion Matrices**.

In [ ]:
models = {'Logistic Regression': lr_model, 'Random Forest': rf_model}
metrics_list = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    
    metrics_list.append({
        'Model': name,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc,
        'PR-AUC': pr_auc,
        'TP': cm[1, 1],
        'FP': cm[0, 1],
        'FN': cm[1, 0]
    })
    
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i], cbar=False,
                xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
    axes[i].set_title(f'Confusion Matrix: {name}', fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(metrics_list).set_index('Model')
display(results_df.style.highlight_max(axis=0, color='#d4edda'))

In [ ]:
# ROC & Precision-Recall Curves Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    precisions, recalls, _ = precision_recall_curve(y_test, y_prob)
    
    ax1.plot(fpr, tpr, lw=2.5, label=f"{name} (ROC-AUC = {roc_auc_score(y_test, y_prob):.4f})")
    ax2.plot(recalls, precisions, lw=2.5, label=f"{name} (PR-AUC = {average_precision_score(y_test, y_prob):.4f})")

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_title('ROC Curves Comparison', fontweight='bold')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate (Recall)')
ax1.legend()

ax2.axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.4f})')
ax2.set_title('Precision-Recall Curves Comparison', fontweight='bold')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend()

plt.tight_layout()
plt.show()

## 🔍 8. Feature Importance Analysis: Key Fraud Predictors
Identifying which features drive the model's fraud predictions.

In [ ]:
rf_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_importances.head(10).values, y=rf_importances.head(10).index, palette='viridis')
plt.title('Top 10 Key Fraud Predictors (Random Forest Importance)', fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('Top 5 Fraud Predictors:')
for rank, (feat, val) in enumerate(rf_importances.head(5).items(), 1):
    print(f'  {rank}. {feat}: {val:.4f}')

## 🎯 9. Conclusion & Key Takeaways
1. **Impact of SMOTE**: Balancing the training set enabled both models to detect rare fraud instances effectively without overfitting the test set.
2. **Algorithm Comparison**: Random Forest achieved exceptional precision and PR-AUC, minimizing false alerts while maintaining high recall.
3. **Domain Insights**: Features **V14, V12, V10, V17** serve as the most potent indicators of fraudulent transactions.